# The Critique Loop (and Why the Naive Version Fails) [Step 04.01]

> **MLCourse - Agentic AI - Agent Patterns**

The idea is irresistibly simple:

```
   draft  -->  critique  -->  revise  -->  better draft
                   ^                            |
                   +----------------------------+
```

Ask the model to write something, ask it what is wrong with it, ask it to fix
that. No extra models, no tools, no training.

It also fails in a specific and well-documented way, and this notebook shows the
failure before the next notebooks fix it.

### What you'll learn

- The plain critique-and-revise loop, implemented.
- The **sycophancy failure**: what happens when you ask "is this good?"
- The three ingredients that make critique work: a *separate* call, a *specific*
  standard, and an *objective* check.
- Measured violation counts before and after, so the improvement is not a vibe.

### Why it matters

Self-critique is one of the few techniques that improves quality without more
data, more tools or a bigger model. But the naive implementation - the one in
every blog post - reliably produces "This response is clear, professional and
addresses the customer's concerns." followed by a revision that changes three
words. Knowing why is the difference between a technique and a ritual.

### Prerequisites

- [01_langchain/02_prompts_and_chains](../../../01_langchain/02_prompts_and_chains)
- [02_langgraph/08_advanced_reasoning_patterns/02_reflexion](../../../02_langgraph/08_advanced_reasoning_patterns/02_reflexion.ipynb) - the same loop, grounded in task failure. Notebook 04 of this module contrasts the two directly.

### Setup: environment, model, token counting, rate-limit-aware call helper


In [ ]:
import os                              # environment variables
import time                            # timing and pacing
import json                            # pretty-printing structured context
from pathlib import Path               # locating the track root
from dotenv import load_dotenv         # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we hit the repo root, then load the
# (gitignored) .env that lives inside 03_agentic_ai. Note the extra path
# segment: the walk-up lands on the REPO ROOT, not on the track folder.
TRACK = Path.cwd()
while not (TRACK / "03_agentic_ai").exists() and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / "03_agentic_ai" / ".env")

GROQ_MODEL = "qwen/qwen3.8-27b"        # hosted, fast, generous free tier
# Local alternative (documented, not used here): Ollama `llama3.1:8b` via
# `from langchain_ollama import ChatOllama`. OpenAI is never used in this course.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 300, **kw):
    """One place that constructs the chat model, so every notebook is identical."""
    return ChatGroq(model=GROQ_MODEL, temperature=temperature,
                    max_tokens=max_tokens, **kw)


# --- Token counting -----------------------------------------------------------
# Two different numbers, and it matters which one you are looking at:
#   * approx_tokens(): a LOCAL estimate using tiktoken's cl100k_base. It is not
#     the model's own tokenizer, so treat it as "within ~10%", good for
#     budgeting BEFORE you send a request.
#   * usage_metadata on the response: the provider's EXACT count. Ground truth,
#     but only available AFTER you have already paid for the call.
import tiktoken

_ENC = tiktoken.get_encoding("cl100k_base")


def approx_tokens(text) -> int:
    """Approximate token count for a string (or anything str()-able)."""
    return len(_ENC.encode(str(text)))


# --- Rate-limit-aware calling --------------------------------------------------
# The Groq free tier allows 8000 tokens per minute. Several notebooks here make
# many small calls in a loop, so we self-pace well under the ceiling and retry
# with exponential backoff if we are throttled anyway.

TPM_BUDGET = 3500                       # deliberately conservative
_WINDOW = []                            # [(timestamp, tokens), ...]
USAGE = {"calls": 0, "in": 0, "out": 0, "seconds": 0.0}


def _pace(cost: int):
    """Sleep just enough that our rolling 60s token usage stays under budget."""
    now = time.time()
    while True:
        recent = [(t, n) for (t, n) in _WINDOW if now - t < 60]
        _WINDOW[:] = recent
        if sum(n for _, n in recent) + cost <= TPM_BUDGET or not recent:
            return
        time.sleep(min(5.0, 60 - (now - recent[0][0]) + 0.5))
        now = time.time()


def chat(messages, llm=None, temperature=0.0, max_tokens=300, retries=5):
    """Send `messages`, return the AIMessage. Paces, retries, and meters usage.

    `messages` is a list of (role, content) tuples or LangChain message objects.
    """
    llm = llm or make_llm(temperature=temperature, max_tokens=max_tokens)
    est = approx_tokens(messages) + max_tokens
    delay = 4.0
    for attempt in range(retries):
        _pace(est)
        t0 = time.time()
        try:
            out = llm.invoke(messages)
        except Exception as exc:
            if "rate_limit" in str(exc) or "429" in str(exc):
                time.sleep(delay)
                delay = min(delay * 2, 45)
                continue
            raise
        u = out.usage_metadata or {}
        _WINDOW.append((time.time(), u.get("total_tokens", est)))
        USAGE["calls"] += 1
        USAGE["in"] += u.get("input_tokens", 0)
        USAGE["out"] += u.get("output_tokens", 0)
        USAGE["seconds"] += time.time() - t0
        return out
    raise RuntimeError("still rate limited after %d attempts" % retries)


def ask(prompt: str, system: str = None, **kw) -> str:
    """Convenience wrapper: one user turn in, plain text out."""
    msgs = ([("system", system)] if system else []) + [("user", prompt)]
    return chat(msgs, **kw).content.strip()


print("model:", GROQ_MODEL)
print("key loaded:", bool(os.getenv("GROQ_API_KEY")))
print("tokenizer:", "cl100k_base (approximation)")


### The shared task


In [ ]:
# One drafting task, used by every notebook in this module so the comparisons are
# apples to apples. It is chosen because it has MANY ways to go subtly wrong -
# which is exactly the situation principles are for.

SITUATION = """A customer, Elena Duarte, has emailed angrily. Her order NW-10261
(SKU TH-275-BLK, supplied by Meridian Components GmbH) was cancelled by us
without warning because the frame size was discontinued. She has been waiting
eleven days. Our records show she was charged and NOT yet refunded; finance says
the refund will clear in 3-5 business days but has occasionally taken longer.
She has asked for compensation. Our policy allows a goodwill voucher of up to
15 EUR, which requires a supervisor's approval that has not yet been given."""

DRAFT_INSTRUCTION = ("Write the reply we should send to Elena. Write only the "
                     "email body, no subject line and no notes.")

print(SITUATION)


### Deterministic checks


In [ ]:
# Not every principle can be checked by code, but several can - and the ones that
# can are worth far more than an LLM's opinion, because they cannot be argued
# with. We use them to MEASURE whether critique-and-revise actually changed
# anything, rather than trusting the model's report of its own improvement.

import re

INTERNAL_TERMS = ["th-275", "meridian", "supplier", "sku", "finance says",
                  "our records show"]
OVERPROMISE = ["guarantee", "guaranteed", "i promise", "we promise", "definitely will",
               "will certainly", "rest assured that you will", "immediately refund"]
EMPATHY = ["sorry", "apolog", "understand", "frustrat", "disappoint", "regret"]


def check(text):
    """Return a dict of objective observations about a draft."""
    low = text.lower()
    words = len(re.findall(r"\b[\w'-]+\b", text))
    return {
        "words": words,
        "over_150_words": words > 150,
        "internal_leaks": [t for t in INTERNAL_TERMS if t in low],
        "overpromises": [t for t in OVERPROMISE if t in low],
        "has_empathy": any(t in low for t in EMPATHY),
        "mentions_voucher": "voucher" in low or "15 eur" in low or "15EUR".lower() in low,
        "states_refund_window": bool(re.search(r"3\s*[-to]+\s*5\s*business days", low)),
    }


def violations(text):
    """Count objective violations. Lower is better."""
    c = check(text)
    n = 0
    n += len(c["internal_leaks"])
    n += len(c["overpromises"])
    n += int(c["over_150_words"])
    n += int(not c["has_empathy"])
    return n


def report(text, label):
    c = check(text)
    print("%-22s words=%-4d leaks=%-22s overpromise=%-18s empathy=%-5s -> %d violations"
          % (label, c["words"], ",".join(c["internal_leaks"]) or "none",
             ",".join(c["overpromises"]) or "none", c["has_empathy"], violations(text)))
    return c


### 1. The draft

A deliberately ordinary prompt, of the sort you would actually write. Watch what
leaks into it.

In [4]:
draft = chat([("user", SITUATION + "\n\n" + DRAFT_INSTRUCTION)],
             temperature=0.3, max_tokens=420).content.strip()
print(draft)
print()
_ = report(draft, "original draft")

Dear Elena,

I am writing to sincerely apologize for the frustration and inconvenience caused by the cancellation of your order NW-10261. I understand that you have been waiting for eleven days without prior notification regarding the discontinuation of the frame size for SKU TH-275-BLK, and I regret that our communication failed to meet your expectations.

Please be assured that your refund is currently being processed. While our finance team indicates that funds typically clear within 3-5 business days, I have flagged your account to ensure this is prioritized. If the funds have not appeared by the end of this week, please let me know immediately so I can escalate the matter further.

Regarding your request for compensation, I have submitted a request for a goodwill voucher of up to 15 EUR to our supervisor for approval. I will follow up with you as soon as I have a definitive answer, which I expect to be within the next 24 hours.

Thank you for your patience while we resolve this.



Whatever the specific numbers above, note the shape of the problem: nothing in
the prompt told the model that "Meridian Components GmbH" is an internal
supplier name, or that promising a refund date it cannot control is dangerous.
The model has no way to know. **It is not making a mistake - it is answering the
question it was asked.**

That is the first insight of this whole module:

> Quality failures are usually **unstated standards**, not model errors.

### 2. The naive critique

Here is the version that does not work. Watch the wording carefully - it is
exactly what people write.

In [5]:
naive_critique = chat([("user", "Here is a customer service email. Is it good? "
                                "How could it be improved?\n\n" + draft)],
                      temperature=0.0, max_tokens=280).content.strip()
print(naive_critique)

This is a **solid, professional, and safe** email. It hits all the necessary notes: it apologizes, explains the situation, confirms the refund, addresses the compensation request, and sets clear expectations for follow-up.

However, it can be improved to sound **less robotic, more empathetic, and more human**. The current tone is very "corporate standard," which can feel cold to a customer who is already frustrated.

Here’s a breakdown of what’s good, what could be better, and a revised version.

### ✅ What’s Good
1.  **Clear Structure:** It’s easy to read and covers all points (apology, refund, compensation, next steps).
2.  **Specifics:** It references the order number, SKU, and specific timeline (11 days), which shows you’ve actually looked into the issue.
3.  **Action-Oriented:** It doesn’t just say "we’re sorry"; it says "I have flagged your account" and "I have submitted a request."
4.  **Clear Deadline:** "Within the next 24 hours" is a concrete promise.

### ⚠️ Areas for Improv

### What went wrong

Three failures, all visible above:

1. **Sycophancy.** "Is it good?" invites agreement. Models are trained to be
   agreeable, so they agree - especially about text they just wrote.
2. **No standard.** "Improved" against *what*? The critique defaults to generic
   writing advice (tone, structure, length) and never mentions the supplier name,
   because nothing told it that mattered.
3. **No consequences.** The critique costs nothing to be wrong about. Nothing
   checks it.

Let us apply the revision anyway and measure whether it changed anything real.

In [6]:
naive_revision = chat([("user", "Rewrite the email applying this feedback.\n\n"
                                "EMAIL:\n%s\n\nFEEDBACK:\n%s\n\n"
                                "Output only the rewritten email body."
                                % (draft, naive_critique))],
                      temperature=0.3, max_tokens=420).content.strip()
print(naive_revision)
print()
_ = report(draft, "original draft")
_ = report(naive_revision, "naive revision")

Subject: Update on your order NW-10261

Hi Elena,

I’m really sorry for the frustration and inconvenience caused by the cancellation of your order. I know you’ve been waiting eleven days without any prior notice about the discontinuation of the frame size (SKU TH-275-BLK), and I completely understand why that’s disappointing. We missed the mark on communication, and I apologize for that.

Good news: your refund is already being processed. I’ve flagged your account to prioritize this, so it should clear within 3-5 business days. If you don’t see the funds by the end of this week, please let me know right away so I can step in and escalate it further.

Regarding your request for compensation, I’ve submitted a proposal for a 15 EUR goodwill voucher for approval. I’ll follow up with you within the next 24 hours with a definitive answer.

Thank you for your patience while we sort this out.

Best regards,

Customer Support Team

original draft         words=173  leaks=th-275,sku             

> **Read the two rows.** The revision is usually *different* - often longer and
> more polished - while the objective violations barely move. Fluency improved;
> the things that would actually get you in trouble did not. This is the single
> most common outcome of naive self-critique, and it is invisible unless you
> measure.

### 3. The three ingredients

What separates critique that works from critique that flatters:

| Ingredient | Naive version | Working version |
|---|---|---|
| **Separate call** | "Write it, then check it" in one prompt | Critique in its own call, with the draft as *input* |
| **Specific standard** | "Is it good?" | A written list of things it must not do |
| **Objective check** | The model's opinion | Code, a test, a checker - anything the model cannot argue with |

The second one is what a **constitution** is, and it is notebook 02. The third
is what our `check()` function does, and it is why this module can report numbers
instead of impressions.

Let us fix ingredient 2 in the crudest possible way - just naming the standards -
and see how far that alone gets us.

In [7]:
NAMED_STANDARDS = """Check the email against these specific requirements and list
EVERY violation you find. For each one, quote the exact offending text.

1. Must not mention internal information: supplier names, SKU codes, internal
   systems, or what our records or finance department say.
2. Must not promise anything we cannot control, including exact refund dates.
3. Must not offer compensation that has not been approved.
4. Must acknowledge the customer's frustration explicitly.
5. Must be under 150 words.

If a requirement is met, say so in one short line. Do not rewrite the email."""

targeted_critique = chat([("user", NAMED_STANDARDS + "\n\nEMAIL:\n" + draft)],
                         temperature=0.0, max_tokens=420).content.strip()
print(targeted_critique)

**Requirement 1: Must not mention internal information**
**VIOLATION.**
*   "order NW-10261" (Internal order/SKU code)
*   "SKU TH-275-BLK" (Internal SKU code)
*   "our finance team indicates" (Internal department/system reference)

**Requirement 2: Must not promise anything we cannot control, including exact refund dates**
**VIOLATION.**
*   "If the funds have not appeared by the end of this week" (Promises a specific timeframe/deadline for an external banking process)

**Requirement 3: Must not offer compensation that has not been approved**
**VIOLATION.**
*   "I have submitted a request for a goodwill voucher of up to 15 EUR to our supervisor for approval." (Offers specific compensation amount before approval is granted)

**Requirement 4: Must acknowledge the customer's frustration explicitly**
**MET.**
*   "I am writing to sincerely apologize for the frustration..."

**Requirement 5: Must be under 150 words**
**MET.**
*   The email is approximately 135 words.


In [8]:
targeted_revision = chat([("user",
                           "Rewrite the email so that every violation listed below is "
                           "fixed. Change nothing else.\n\nEMAIL:\n%s\n\n"
                           "VIOLATIONS:\n%s\n\nOutput only the rewritten email body."
                           % (draft, targeted_critique))],
                         temperature=0.2, max_tokens=420).content.strip()
print(targeted_revision)
print()
_ = report(draft, "original draft")
_ = report(naive_revision, "naive revision")
_ = report(targeted_revision, "targeted revision")

Dear Elena,

I am writing to sincerely apologize for the frustration and inconvenience caused by the cancellation of your order. I understand that you have been waiting for eleven days without prior notification regarding the discontinuation of the frame size, and I regret that our communication failed to meet your expectations.

Please be assured that your refund is currently being processed. Funds typically clear within 3-5 business days. I have flagged your account to ensure this is prioritized. If the funds have not appeared after this period, please let me know immediately so I can escalate the matter further.

Regarding your request for compensation, I am reviewing options for a goodwill gesture. I will follow up with you as soon as I have a definitive answer, which I expect to be within the next 24 hours.

Thank you for your patience while we resolve this.

Sincerely,

Customer Support Team

original draft         words=173  leaks=th-275,sku             overpromise=none         

In [9]:
v0, v1, v2 = violations(draft), violations(naive_revision), violations(targeted_revision)
print("MEASURED, this run, %s:" % GROQ_MODEL)
print("  original draft     : %d objective violations" % v0)
print("  naive critique     : %d  (%+d)" % (v1, v1 - v0))
print("  targeted critique  : %d  (%+d)" % (v2, v2 - v0))
print()
if v2 < v1:
    print("Naming the standards is doing the work. Same number of LLM calls,")
    print("same model, same task - only the critique prompt changed.")
elif v2 == v1 == v0:
    print("Neither critique changed the objective violations on this run. Note that")
    print("this is itself the lesson: a critique loop that produces a nicer-sounding")
    print("email while leaving every checkable problem in place has cost you two")
    print("extra LLM calls and bought nothing.")
else:
    print("The results are mixed on this single run. Single-run comparisons of a")
    print("stochastic system are weak evidence - notebook 03 runs the full loop and")
    print("reports across iterations.")

MEASURED, this run, qwen/qwen3.8-27b:
  original draft     : 3 objective violations
  naive critique     : 3  (+0)
  targeted critique  : 0  (-3)

Naming the standards is doing the work. Same number of LLM calls,
same model, same task - only the critique prompt changed.


### 4. Pitfalls

- **Critique in the same call as the draft.** "Write it, then critique it"
  produces a critique that defends the draft it just wrote.
- **Asking "is this good?"** The answer is yes. Always ask "which of these
  specific rules does it break?"
- **Trusting the critique's self-report.** "I fixed all the issues" is a
  generated sentence, not a fact. Check.
- **No objective anchor.** If nothing in your loop can say "no", the loop cannot
  converge - it can only churn.
- **Unbounded loops.** Always cap iterations. Two or three is usually all you get.

### Recap

| Idea | Takeaway |
|---|---|
| Draft -> critique -> revise | Simple, and simple to get wrong |
| Sycophancy | "Is it good?" always answers yes |
| Unstated standards | Quality failures are usually missing rules, not model errors |
| Name the rules | The cheapest fix available, measured above |
| Check objectively | Otherwise you measure fluency, not correctness |

**Next:** [02_writing_a_constitution](02_writing_a_constitution.ipynb) - turning
that ad-hoc list of standards into a real, reusable principle set.